# Used Bike Price Analysis

Notebook containing the Python code extracted from the supplied text. The dataset is expected to be available as `bikes.csv` in the notebook's working directory.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries loaded.")

In [ ]:
df_raw = pd.read_csv("bikes.csv")

print("Dataset shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())

display(df_raw.head())

### Cell 2: Load and inspect the dataset

In [ ]:
print("Data types:")
print(df_raw.dtypes)

print("\nMissing values:")
print(df_raw.isnull().sum())

print("\nDuplicate rows:", df_raw.duplicated().sum())

if "price" in df_raw.columns:
    print("\nPrice summary:")
    print(df_raw["price"].describe())

### Cell 3: Check data quality

In [ ]:
df = df_raw.copy()

# Convert common numeric columns to numbers.
# Text such as '35 kmpl' or '25,000 km' will be handled below.
def get_number(value):
    if pd.isna(value):
        return np.nan
    value = str(value).replace(",", "")
    number = ""
    dot_used = False

    for char in value:
        if char.isdigit():
            number += char
        elif char == "." and not dot_used:
            number += char
            dot_used = True
        elif number:
            break

    return float(number) if number else np.nan


for col in ["price", "model_year", "kms_driven", "mileage", "power"]:
    if col in df.columns:
        df[col] = df[col].apply(get_number)

print("Numeric columns converted where possible.")
display(df.head())

### Cell 4: Clean numeric columns

In [ ]:
rows_before = len(df)
duplicates = df.duplicated().sum()
df = df.drop_duplicates()

# Remove rows without a valid positive price.
df = df.dropna(subset=["price"])
df = df[df["price"] > 0]

# Remove impossible model years if the column exists.
if "model_year" in df.columns:
    current_year = pd.Timestamp.today().year
    df = df[df["model_year"].between(1990, current_year)]

# Remove impossible negative values.
for col in ["kms_driven", "mileage", "power"]:
    if col in df.columns:
        df.loc[df[col] < 0, col] = np.nan

print("Rows before cleaning:", rows_before)
print("Duplicate rows removed:", duplicates)
print("Rows after basic cleaning:", len(df))
print("Rows removed:", rows_before - len(df))

### Cell 5: Remove duplicates and invalid prices

In [ ]:
if "model_year" in df.columns:
    current_year = pd.Timestamp.today().year
    df["bike_age"] = current_year - df["model_year"]

# Keep missing values in numerical predictors for now.
# The model pipeline will impute them after the train/test split.
if "location" in df.columns:
    df["location"] = df["location"].fillna("Unknown")

print("Missing values after basic cleaning:")
print(df.isnull().sum())

### Cell 6: Create bike age and handle missing values

In [ ]:
df_clean = df.copy()
df_clean.to_csv("bikes_cleaned.csv", index=False)

print("Cleaned dataset saved as bikes_cleaned.csv")
display(df_clean.head())

### Cell 7: Save cleaned dataset

## Q2. Market Pricing Analysis (EDA)

In [ ]:
print("Price statistics:")
display(df_clean["price"].describe())

print("Mean price: ₹{:,.0f}".format(df_clean["price"].mean()))
print("Median price: ₹{:,.0f}".format(df_clean["price"].median()))
print("Minimum price: ₹{:,.0f}".format(df_clean["price"].min()))
print("Maximum price: ₹{:,.0f}".format(df_clean["price"].max()))
print("Price skewness:", round(df_clean["price"].skew(), 2))

The sales team wants to understand which bike characteristics may be associated with resale price. All observations below should be written after looking at the output generated by the cells.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df_clean, x="price", kde=True)
plt.title("Distribution of Used Bike Prices")
plt.xlabel("Price")
plt.show()

plt.figure(figsize=(9, 3))
sns.boxplot(data=df_clean, x="price")
plt.title("Potential Price Outliers")
plt.xlabel("Price")
plt.show()

### Cell 9: Price distribution and outliers

In [ ]:
# Extract the first word as a simple brand approximation if brand is not provided.
if "brand" not in df_clean.columns and "model_name" in df_clean.columns:
    df_clean["brand"] = df_clean["model_name"].astype(str).str.split().str[0]

if "brand" in df_clean.columns:
    brand_stats = (
        df_clean.groupby("brand")["price"]
        .agg(["median", "mean", "count"])
        .reset_index()
    )

    # Avoid comparing brands represented by only a few listings.
    brand_stats = brand_stats[brand_stats["count"] >= 10]
    brand_stats = brand_stats.sort_values("median", ascending=False)

    display(brand_stats.head(10))

    top_brands = brand_stats.head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_brands, x="median", y="brand")
    plt.title("Top Brands by Median Listed Price")
    plt.xlabel("Median price")
    plt.ylabel("Brand")
    plt.show()
else:
    print("Brand analysis skipped: model_name/brand column not found.")

### Cell 10: Brand-level price comparison

In [ ]:
if "kms_driven" in df_clean.columns:
    print(
        "Correlation between kilometres and price:",
        round(df_clean[["kms_driven", "price"]].corr().iloc[0, 1], 3)
    )

    plot_data = df_clean.dropna(subset=["kms_driven", "price"])
    if len(plot_data) > 3000:
        plot_data = plot_data.sample(3000, random_state=42)

    plt.figure(figsize=(9, 5))
    sns.scatterplot(data=plot_data, x="kms_driven", y="price", alpha=0.5)
    plt.title("Kilometres Driven vs Bike Price")
    plt.show()

### Cell 11: Kilometres driven versus price

In [ ]:
if "bike_age" in df_clean.columns:
    plot_data = df_clean.dropna(subset=["bike_age", "price"])
    plot_data = plot_data[plot_data["bike_age"].between(0, 40)]

    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=plot_data, x="bike_age", y="price", alpha=0.5)
    plt.title("Bike Age vs Bike Price")
    plt.show()

    print(
        "Correlation between bike age and price:",
        round(df_clean[["bike_age", "price"]].corr().iloc[0, 1], 3)
    )

### Cell 12: Bike age versus price

In [ ]:
numeric_df = df_clean.select_dtypes(include="number")

plt.figure(figsize=(10, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

print("Correlation of numeric features with price:")
print(numeric_df.corr()["price"].sort_values(ascending=False))

### Cell 13: Correlation heatmap

In [ ]:
q1 = df_clean["price"].quantile(0.25)
q3 = df_clean["price"].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr

high_price_bikes = df_clean[df_clean["price"] > upper_limit]

print("IQR upper limit: ₹{:,.0f}".format(upper_limit))
print("Number of high-price records:", len(high_price_bikes))

cols_to_show = [c for c in ["model_name", "brand", "model_year", "power", "price"] if c in df_clean.columns]
display(high_price_bikes[cols_to_show].head(15))

### Cell 14: Investigate high-price outliers

### EDA findings to write

## Q3. Feature Engineering and Variable Selection

In [ ]:
# Use only columns that actually exist in this dataset.
numeric_candidates = ["bike_age", "kms_driven", "power", "mileage"]
categorical_candidates = ["owner", "brand", "location"]

numeric_features = [c for c in numeric_candidates if c in df_clean.columns]
categorical_features = [c for c in categorical_candidates if c in df_clean.columns]

features = numeric_features + categorical_features

if not features:
    raise ValueError("No usable predictor columns found. Check the dataset columns.")

df_model = df_clean[features + ["price"]].copy()
X = df_model[features]
y = df_model["price"]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Model dataset shape:", df_model.shape)

Choose features based on EDA and business understanding. Do not include both model_year and bike_age in the same model because they contain essentially the same information. Prepare model features

In [ ]:
numeric_for_corr = [c for c in numeric_features if c in df_model.columns]

if numeric_for_corr:
    display(df_model[numeric_for_corr + ["price"]].corr())

    print("Feature correlation with price:")
    print(df_model[numeric_for_corr + ["price"]].corr()["price"].sort_values())

### Check numerical relationships and correlations

## Q4. Build a Bike Price Estimation Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

We will use an 80/20 train-test split. Imputation and one-hot encoding are fitted on the training data through a pipeline to reduce data leakage.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

transformers = []
if numeric_features:
    transformers.append(("numeric", numeric_pipeline, numeric_features))
if categorical_features:
    transformers.append(("categorical", categorical_pipeline, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regression", LinearRegression())
])

model.fit(X_train, y_train)
print("Linear Regression model trained.")

### Cell 18: Build preprocessing and Linear Regression pipeline

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "actual_price": y_test.values,
    "predicted_price": np.round(y_pred, 0),
    "error_actual_minus_predicted": np.round(y_test.values - y_pred, 0)
})
comparison["absolute_error"] = comparison["error_actual_minus_predicted"].abs()

display(comparison.head(15))

### Cell 19: Predict and compare actual prices

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE: ₹{:,.0f}".format(mae))
print("MSE: {:,.0f}".format(mse))
print("RMSE: ₹{:,.0f}".format(rmse))
print("R²: {:.3f}".format(r2))

## Q5. Model Validation and Business Recommendation

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.5)
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], "r--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Bike Prices")
plt.show()

residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residuals vs Predicted Price")
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True)
plt.title("Residual Distribution")
plt.xlabel("Residual")
plt.show()

print("Mean residual:", round(residuals.mean(), 2))

### Actual versus predicted prices

In [ ]:
# Q-Q plot: points near the line suggest residuals are closer to normal.
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# A correlation between predicted price and absolute residuals can indicate
# that error size changes with predicted price. This is a diagnostic, not a formal test.
from scipy.stats import spearmanr

corr, p_value = spearmanr(y_pred, np.abs(residuals))
print("Correlation: predicted price vs absolute residual:", round(corr, 3))
print("p-value:", round(p_value, 5))

### Cell 23: Check residual normality and changing variance

In [ ]:
comparison["percentage_error"] = (
    comparison["error_actual_minus_predicted"] / comparison["actual_price"] * 100
)

print("Largest underestimates (actual price is higher than prediction):")
display(comparison.nlargest(5, "error_actual_minus_predicted"))

print("Largest overestimates (prediction is higher than actual price):")
display(comparison.nsmallest(5, "error_actual_minus_predicted"))

### Cell 24: Largest prediction errors

In [ ]:
comparison.to_csv("bike_price_predictions.csv", index=False)

import joblib
joblib.dump(model, "bike_price_linear_regression.joblib")

print("Saved bike_price_predictions.csv")
print("Saved bike_price_linear_regression.joblib")

### Cell 25: Save outputs